In [ ]:
# --- standard library imports ---
import numpy as np
import pandas as pd
from pathlib import Path

from ase import Atoms
from ase.io import read, Trajectory
from ase.visualize import view
from pymatgen.io.ase import AseAtomsAdaptor

# --- IMPORTS FROM THIS PROJECT ---
from gnome_dreams_oer_screening import OER_catalyst_study
from gnome_dreams_oer_screening import Explog

In [23]:
EXPLOG = Explog()
EXPLOG.init('TEMP_vasp_calcs', 'test')

In [24]:
# Test structures, to hand to the study object:
ZrO2_m = read('ZrO2_m.cif')
TiO2_r = read('TiO2_rutile.cif')
test_cands = Trajectory('test_cands.traj')

atoms = test_cands[18]
S = AseAtomsAdaptor.get_structure(atoms)

In [25]:
# Dummy setup of frist row (for testing):
study = OER_catalyst_study(atoms,
                            H2O_gas_free_energy = -14.217,
                            H2_gas_free_energy = -6.77
                           )
cant_id = "07286qnfd"

EXPLOG.add_candidate(
    candidate_id=cant_id,
    reason_or_hypothesis="Ir sites should be great", 
    notes="this is a great candidate", 
    study_obj=study
)

EXPLOG.relational_frame.candidates.df

,candidate_id,reason_or_hypothesis,notes,study_obj,OHDone,idealOverPotential
0,07286qnfd,Ir sites should be great,this is a great candidate,<gnome_dreams_oer_screening.oer.oer_study.OER_...,<NA>,<NA>


In [26]:
p_id = EXPLOG.add_process(candidate_id=cant_id, 
                   calculation_type="bulk_relaxation")
EXPLOG.relational_frame.processes.df

,process_id,candidate_id,job_type,slurmID,status,termination_index,site_index,VASP_dir,processNote
0,0,07286qnfd,bulk_relaxation,<NA>,un-submitted,None,None,TEMP_vasp_calcs\20260211_165549_07286qnfd_bulk...,None


In [27]:
EXPLOG.submit_process(process_id=p_id)
EXPLOG.relational_frame.processes.df

,process_id,candidate_id,job_type,slurmID,status,termination_index,site_index,VASP_dir,processNote
0,0,07286qnfd,bulk_relaxation,<NA>,submitted,None,None,TEMP_vasp_calcs\20260211_165549_07286qnfd_bulk...,None


In [28]:
updates_dict = EXPLOG.update_log()
updates_dict # dict with process_id as key and new status as value

{'0': 'completed'}

In [29]:
EXPLOG.relational_frame.processes.df

,process_id,candidate_id,job_type,slurmID,status,termination_index,site_index,VASP_dir,processNote
0,0,07286qnfd,bulk_relaxation,<NA>,completed,None,None,TEMP_vasp_calcs\20260211_165549_07286qnfd_bulk...,None


In [30]:
study = EXPLOG.relational_frame.candidates[cant_id].study_obj
ranking = study.get_termination_rankings()
if ranking is None:
    study.predict_most_likely_surfaces()
    ranking = study.get_termination_rankings()

ranking.sort_values('Normalized score', ascending=False)

,Miller indices,Termination index,Score,Normalized score
0,"(1, 1, 1)",0,-0.132885,1.000000
1,"(1, 1, 1)",1,-0.132885,1.000000
13,"(0, 1, 0)",13,-0.149823,0.801596
12,"(0, 1, 0)",12,-0.149823,0.801596
7,"(1, 0, -1)",7,-0.163368,0.642931
6,"(1, 0, -1)",6,-0.163368,0.642931
11,"(1, -1, 0)",11,-0.172529,0.535628
10,"(1, -1, 0)",10,-0.172529,0.535628
3,"(1, 1, 0)",3,-0.209062,0.107694
2,"(1, 1, 0)",2,-0.209062,0.107694


In [31]:
study.initialize_oer_surface_study(0)

In [32]:
p_id = EXPLOG.add_process( 
    candidate_id = cant_id,
    calculation_type = "surface_relaxation",
    termination_index = 0
)
# test if indecies are provided for surface relaxations etc....
EXPLOG.relational_frame.processes.df

,process_id,candidate_id,job_type,slurmID,status,termination_index,site_index,VASP_dir,processNote
0,0,07286qnfd,bulk_relaxation,<NA>,completed,None,None,TEMP_vasp_calcs\20260211_165549_07286qnfd_bulk...,None
1,1,07286qnfd,surface_relaxation,<NA>,un-submitted,0,None,TEMP_vasp_calcs\20260211_165549_07286qnfd_surf...,None


In [33]:
EXPLOG.submit_process(process_id = p_id)
EXPLOG.relational_frame.processes.df

,process_id,candidate_id,job_type,slurmID,status,termination_index,site_index,VASP_dir,processNote
0,0,07286qnfd,bulk_relaxation,<NA>,completed,None,None,TEMP_vasp_calcs\20260211_165549_07286qnfd_bulk...,None
1,1,07286qnfd,surface_relaxation,<NA>,submitted,0,None,TEMP_vasp_calcs\20260211_165549_07286qnfd_surf...,None


In [34]:
EXPLOG.update_log()
EXPLOG.relational_frame.processes.df

,process_id,candidate_id,job_type,slurmID,status,termination_index,site_index,VASP_dir,processNote
0,0,07286qnfd,bulk_relaxation,<NA>,completed,None,None,TEMP_vasp_calcs\20260211_165549_07286qnfd_bulk...,None
1,1,07286qnfd,surface_relaxation,<NA>,completed,0,None,TEMP_vasp_calcs\20260211_165549_07286qnfd_surf...,None


In [35]:
study = EXPLOG.relational_frame.candidates.df['study_obj'][0]
surface_study = study.get_surface_studies()[0]

surface_study.determine_adsorption_sites()
surface_study.add_sites_df_relaxed

,site type,add site element,position,G_approx(O),G(O),G_approx(OH),G(OH),G_approx(OOH),G(OOH)
0,on-top,Bi,"[0.8738251530472934, 3.7603148722832302, 26.71...",None,None,None,None,None,None
1,on-top,Rh,"[4.434302668592073, 3.755807610061, 27.1025326...",None,None,None,None,None,None
2,on-top,Rh,"[2.537765277159939, 0.612566311834986, 27.1025...",None,None,None,None,None,None
3,on-top,Rh,"[6.090772638968173, 0.6125663118349847, 27.549...",None,None,None,None,None,None


In [36]:
study = EXPLOG.relational_frame.candidates.df['study_obj'][0]
surface_study = study.get_surface_studies()[0]

surface_study.initialize_adsorption_site_study(0)

p_id = EXPLOG.add_process( 
    candidate_id = cant_id,
    calculation_type = "O_adsorption",
    termination_index = 0,
    site_index = 0
)
EXPLOG.relational_frame.processes.df

,process_id,candidate_id,job_type,slurmID,status,termination_index,site_index,VASP_dir,processNote
0,0,07286qnfd,bulk_relaxation,<NA>,completed,None,None,TEMP_vasp_calcs\20260211_165549_07286qnfd_bulk...,None
1,1,07286qnfd,surface_relaxation,<NA>,completed,0,None,TEMP_vasp_calcs\20260211_165549_07286qnfd_surf...,None
2,2,07286qnfd,O_adsorption,<NA>,un-submitted,0,0,TEMP_vasp_calcs\20260211_165549_07286qnfd_O_ad...,None


In [37]:
study = EXPLOG.relational_frame.candidates.df['study_obj'][0]
surface_study = study.get_surface_studies()[0]
EXPLOG.submit_process(process_id = p_id)

EXPLOG.relational_frame.processes.df

,process_id,candidate_id,job_type,slurmID,status,termination_index,site_index,VASP_dir,processNote
0,0,07286qnfd,bulk_relaxation,<NA>,completed,None,None,TEMP_vasp_calcs\20260211_165549_07286qnfd_bulk...,None
1,1,07286qnfd,surface_relaxation,<NA>,completed,0,None,TEMP_vasp_calcs\20260211_165549_07286qnfd_surf...,None
2,2,07286qnfd,O_adsorption,<NA>,submitted,0,0,TEMP_vasp_calcs\20260211_165549_07286qnfd_O_ad...,None


In [38]:
EXPLOG.update_log()
EXPLOG.relational_frame.processes.df

{'H2O_gas_free_energy': -14.217, 'H2_gas_free_energy': -6.77, 'O2_gas_free_energy': None, 'O_ad_correction': 0.07, 'OH_ad_correction': 0.361, 'OOH_ad_correction': 0.431}
-6.77


,process_id,candidate_id,job_type,slurmID,status,termination_index,site_index,VASP_dir,processNote
0,0,07286qnfd,bulk_relaxation,<NA>,completed,None,None,TEMP_vasp_calcs\20260211_165549_07286qnfd_bulk...,None
1,1,07286qnfd,surface_relaxation,<NA>,completed,0,None,TEMP_vasp_calcs\20260211_165549_07286qnfd_surf...,None
2,2,07286qnfd,O_adsorption,<NA>,completed,0,0,TEMP_vasp_calcs\20260211_165549_07286qnfd_O_ad...,None


In [39]:
EXPLOG.relational_frame.candidates.df

,candidate_id,reason_or_hypothesis,notes,study_obj,OHDone,idealOverPotential
0,07286qnfd,Ir sites should be great,this is a great candidate,<gnome_dreams_oer_screening.oer.oer_study.OER_...,<NA>,2.07


In [40]:
surface_study.update_adsorption_energies()
surface_study.get_adsorption_sites_df()

{'H2O_gas_free_energy': -14.217, 'H2_gas_free_energy': -6.77, 'O2_gas_free_energy': None, 'O_ad_correction': 0.07, 'OH_ad_correction': 0.361, 'OOH_ad_correction': 0.431}
-6.77


,site type,add site element,position,G_approx(O),G(O),G_approx(OH),G(OH),G_approx(OOH),G(OOH)
0,on-top,Bi,"[0.8738251530472934, 3.7603148722832302, 26.71...",0.394198,None,None,None,None,None
1,on-top,Rh,"[4.434302668592073, 3.755807610061, 27.1025326...",None,None,None,None,None,None
2,on-top,Rh,"[2.537765277159939, 0.612566311834986, 27.1025...",None,None,None,None,None,None
3,on-top,Rh,"[6.090772638968173, 0.6125663118349847, 27.549...",None,None,None,None,None,None


In [41]:
EXPLOG.relational_frame.candidates.df 

,candidate_id,reason_or_hypothesis,notes,study_obj,OHDone,idealOverPotential
0,07286qnfd,Ir sites should be great,this is a great candidate,<gnome_dreams_oer_screening.oer.oer_study.OER_...,<NA>,2.07
